# Alpaca-py options trading basic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alpacahq/alpaca-py/blob/master/examples/options/options-trading-basic.ipynb)

- This notebook shows how to use alpaca-py with options trading API endpoints
- Please use ``paper account``. Please ``DO NOT`` use this notebook with live account. In this notebook, we place orders for options as an example.

In [1]:
# Please change the following to your own PAPER api key and secret
# or set them as environment variables (ALPACA_API_KEY, ALPACA_SECRET_KEY).
# You can get them from https://alpaca.markets/

api_key = None
secret_key = None

#### We use paper environment for this example ####
paper = True # Please do not modify this. This example is for paper trading only.
####

# Below are the variables for development this documents
# Please do not change these variables

trade_api_url = None
trade_api_wss = None
data_api_url = None
option_stream_data_wss = None

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


if api_key is None:
    api_key = os.environ.get('ALPACA_API_KEY')

if secret_key is None:
    secret_key = os.environ.get('ALPACA_SECRET_KEY')

In [3]:
# install alpaca-py if it is not available
try:
    import alpaca
except ImportError:
    !python3 -m pip install alpaca-py
    import alpaca

In [6]:
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from alpaca.trading.client import TradingClient
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.historical.option import OptionHistoricalDataClient
from alpaca.trading.stream import TradingStream
from alpaca.data.live.option import OptionDataStream

from alpaca.data.requests import (
    OptionBarsRequest,
    OptionTradesRequest,
    OptionLatestQuoteRequest,
    OptionLatestTradeRequest,
    OptionSnapshotRequest,
    OptionChainRequest    
)
from alpaca.trading.requests import (
    GetOptionContractsRequest,
    GetAssetsRequest,
    MarketOrderRequest,
    GetOrdersRequest,
    ClosePositionRequest
)
from alpaca.trading.enums import (
    AssetStatus,
    ExerciseStyle,
    OrderSide,
    OrderType,
    TimeInForce,
    QueryOrderStatus 
)
from alpaca.common.exceptions import APIError

In [7]:
# to run async code in jupyter notebook
import nest_asyncio
nest_asyncio.apply()

In [8]:
# check version of alpaca-py
alpaca.__version__

'0.43.2'

# Trading Client

In [9]:
# setup clients
trade_client = TradingClient(api_key=api_key, secret_key=secret_key, paper=paper, url_override=trade_api_url)

In [10]:
# check trading account
# There are trhee new columns in the account object:
# - options_buying_power
# - options_approved_level
# - options_trading_level
acct = trade_client.get_account()
acct

{   'account_blocked': False,
    'account_number': 'PA3ZPB2W362P',
    'accrued_fees': '0',
    'buying_power': '190826.89',
    'cash': '99212.25',
    'created_at': datetime.datetime(2025, 2, 7, 1, 47, 10, 815394, tzinfo=TzInfo(UTC)),
    'crypto_status': <AccountStatus.ACTIVE: 'ACTIVE'>,
    'currency': 'USD',
    'daytrade_count': 0,
    'daytrading_buying_power': '0',
    'equity': '102916.64',
    'id': UUID('94ccd725-6370-4308-b880-31e14648310a'),
    'initial_margin': '7601.19',
    'last_equity': '102921.77478568222',
    'last_maintenance_margin': '6839.86',
    'long_market_value': '3802.39',
    'maintenance_margin': '6840.72',
    'multiplier': '2',
    'non_marginable_buying_power': '93512.25',
    'options_approved_level': 3,
    'options_buying_power': '95413.44',
    'options_trading_level': 3,
    'pattern_day_trader': False,
    'pending_transfer_in': None,
    'pending_transfer_out': None,
    'portfolio_value': '102916.64',
    'regt_buying_power': '190826.89',
  

In [11]:
# check account configuration
# - we have new field `max_options_trading_level`
acct_config = trade_client.get_account_configurations()
acct_config

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': True,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

In [12]:
# get list of assets which are options enabled
# - we can filter assets by `options_enabled` attribute
# - asset object has `options_enabled` attribute if it is options enabled
req = GetAssetsRequest(
  attributes = "options_enabled"
)
assets = trade_client.get_all_assets(req)
assets[:2]

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': ['fractional_eh_enabled', 'has_options', 'overnight_halted'],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.OTC: 'OTC'>,
     'fractionable': True,
     'id': UUID('c8720069-e02d-448c-9e7c-f808db900df6'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min_trade_increment': None,
     'name': 'OFFICE PPTYS INCOME TRCOM SHS BEN INT',
     'price_increment': None,
     'shortable': False,
     'status': <AssetStatus.ACTIVE: 'active'>,
     'symbol': 'OPITQ',
     'tradable': False},
 {   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': ['has_options'],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.OTC: 'OTC'>,
     'fractionable': False,
     'id': UUID('162c893c-c085-4862-9311-e98633dda8e7'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min

In [13]:
# get list of options contracts for the given underlying symbol (e.g. SPY,AAPL)
# - get_option_contracts() is a new method to get list of options contracts
# - in this example, we get 2 options contracts for SPY,AAPL
# - you can continue to fetch options contracts by specifying page_token from next_page_token of response
underlying_symbols = ["SPY", "AAPL"]
req = GetOptionContractsRequest(
    underlying_symbols = underlying_symbols,               # specify underlying symbols
    status = AssetStatus.ACTIVE,                           # specify asset status: active (default)
    expiration_date = None,                                # specify expiration date (specified date + 1 day range)
    expiration_date_gte = None,                            # we can pass date object
    expiration_date_lte = None,                            # or string (YYYY-MM-DD)
    root_symbol = None,                                    # specify root symbol
    type = None,                                           # specify option type (ContractType.CALL or ContractType.PUT)
    style = None,                                          # specify option style (ContractStyle.AMERICAN or ContractStyle.EUROPEAN)
    strike_price_gte = None,                               # specify strike price range
    strike_price_lte = None,                               # specify strike price range
    limit = 2,                                             # specify limit
    page_token = None,                                     # specify page token
)
res = trade_client.get_option_contracts(req)
res

{   'next_page_token': 'Mg==',
    'option_contracts': [   {   'close_price': None,
                                'close_price_date': None,
                                'expiration_date': datetime.date(2026, 2, 9),
                                'id': '1e5fcc15-54ea-48c8-a45a-c1b4be7ba1d9',
                                'name': 'AAPL Feb 09 2026 175 Call',
                                'open_interest': None,
                                'open_interest_date': None,
                                'root_symbol': 'AAPL',
                                'size': '100',
                                'status': <AssetStatus.ACTIVE: 'active'>,
                                'strike_price': 175.0,
                                'style': <ExerciseStyle.AMERICAN: 'american'>,
                                'symbol': 'AAPL260209C00175000',
                                'tradable': True,
                                'type': <ContractType.CALL: 'call'>,
                        

In [14]:
# continue to fetch option contracts if there is next_page_token in response
if res.next_page_token is not None:
    req = GetOptionContractsRequest(
        underlying_symbols = underlying_symbols,               # specify underlying symbols
        status = AssetStatus.ACTIVE,                           # specify asset status: active (default)
        expiration_date = None,                                # specify expiration date (specified date + 1 day range)
        expiration_date_gte = None,                            # we can pass date object
        expiration_date_lte = None,                            # or string (YYYY-MM-DD)
        root_symbol = None,                                    # specify root symbol
        type = None,                                           # specify option type (ContractType.CALL or ContractType.PUT)
        style = None,                                          # specify option style (ContractStyle.AMERICAN or ContractStyle.EUROPEAN)
        strike_price_gte = None,                               # specify strike price range
        strike_price_lte = None,                               # specify strike price range
        limit = 2,                                             # specify limit
        page_token = res.next_page_token,                      # specify page token
    )
    res = trade_client.get_option_contracts(req)
    display(res)

{   'next_page_token': 'NA==',
    'option_contracts': [   {   'close_price': None,
                                'close_price_date': None,
                                'expiration_date': datetime.date(2026, 2, 9),
                                'id': '3890041b-f4fa-462c-99c4-e160e0b59439',
                                'name': 'AAPL Feb 09 2026 185 Call',
                                'open_interest': None,
                                'open_interest_date': None,
                                'root_symbol': 'AAPL',
                                'size': '100',
                                'status': <AssetStatus.ACTIVE: 'active'>,
                                'strike_price': 185.0,
                                'style': <ExerciseStyle.AMERICAN: 'american'>,
                                'symbol': 'AAPL260209C00185000',
                                'tradable': True,
                                'type': <ContractType.CALL: 'call'>,
                        

In [15]:
# get options contract by symbol
# - get_option_contract() is a new method to get options contract by symbol or id
symbol = res.option_contracts[0].symbol
contract = trade_client.get_option_contract(symbol)
contract

{   'close_price': None,
    'close_price_date': None,
    'expiration_date': datetime.date(2026, 2, 9),
    'id': '3890041b-f4fa-462c-99c4-e160e0b59439',
    'name': 'AAPL Feb 09 2026 185 Call',
    'open_interest': None,
    'open_interest_date': None,
    'root_symbol': 'AAPL',
    'size': '100',
    'status': <AssetStatus.ACTIVE: 'active'>,
    'strike_price': 185.0,
    'style': <ExerciseStyle.AMERICAN: 'american'>,
    'symbol': 'AAPL260209C00185000',
    'tradable': True,
    'type': <ContractType.CALL: 'call'>,
    'underlying_asset_id': UUID('b0b6dd9d-8b9b-48a9-ba46-b9d54906e415'),
    'underlying_symbol': 'AAPL'}

In [16]:
# get options contract by id
id = res.option_contracts[0].id
contract = trade_client.get_option_contract(symbol_or_id=id)
contract

{   'close_price': None,
    'close_price_date': None,
    'expiration_date': datetime.date(2026, 2, 9),
    'id': '3890041b-f4fa-462c-99c4-e160e0b59439',
    'name': 'AAPL Feb 09 2026 185 Call',
    'open_interest': None,
    'open_interest_date': None,
    'root_symbol': 'AAPL',
    'size': '100',
    'status': <AssetStatus.ACTIVE: 'active'>,
    'strike_price': 185.0,
    'style': <ExerciseStyle.AMERICAN: 'american'>,
    'symbol': 'AAPL260209C00185000',
    'tradable': True,
    'type': <ContractType.CALL: 'call'>,
    'underlying_asset_id': UUID('b0b6dd9d-8b9b-48a9-ba46-b9d54906e415'),
    'underlying_symbol': 'AAPL'}

In [17]:
# get put options contracts
underlying_symbols = ["SPY"]

# specify expiration date range
now = datetime.now(tz = ZoneInfo("America/New_York"))
day1 = now + timedelta(days = 1)
day60 = now + timedelta(days = 60)

req = GetOptionContractsRequest(
    underlying_symbols = underlying_symbols,                     # specify underlying symbols
    status = AssetStatus.ACTIVE,                                 # specify asset status: active (default)
    expiration_date = None,                                      # specify expiration date (specified date + 1 day range)
    expiration_date_gte = day1.date(),                           # we can pass date object
    expiration_date_lte = day60.strftime(format = "%Y-%m-%d"),   # or string
    root_symbol = None,                                          # specify root symbol
    type = "put",                                                # specify option type: put
    style = ExerciseStyle.AMERICAN,                              # specify option style: american
    strike_price_gte = None,                                     # specify strike price range
    strike_price_lte = None,                                     # specify strike price range
    limit = 100,                                                 # specify limit
    page_token = None,                                           # specify page
)
res = trade_client.get_option_contracts(req)
res.option_contracts[:2]

[{   'close_price': '0.01',
     'close_price_date': datetime.date(2026, 2, 4),
     'expiration_date': datetime.date(2026, 2, 10),
     'id': '279cde75-79c6-4285-9d74-a8b450ee55fb',
     'name': 'SPY Feb 10 2026 500 Put',
     'open_interest': '270',
     'open_interest_date': datetime.date(2026, 2, 5),
     'root_symbol': 'SPY',
     'size': '100',
     'status': <AssetStatus.ACTIVE: 'active'>,
     'strike_price': 500.0,
     'style': <ExerciseStyle.AMERICAN: 'american'>,
     'symbol': 'SPY260210P00500000',
     'tradable': True,
     'type': <ContractType.PUT: 'put'>,
     'underlying_asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'underlying_symbol': 'SPY'},
 {   'close_price': '0.01',
     'close_price_date': datetime.date(2026, 1, 30),
     'expiration_date': datetime.date(2026, 2, 10),
     'id': '8ab18b30-4edb-4ab4-9017-6006b40b4ea5',
     'name': 'SPY Feb 10 2026 505 Put',
     'open_interest': '1',
     'open_interest_date': datetime.date(2026, 2, 5),
     'r

In [18]:
# get high open_interest contract
open_interest = 0
high_open_interest_contract = None
for contract in res.option_contracts:
    if (contract.open_interest is not None) and (int(contract.open_interest) > open_interest):
        open_interest = int(contract.open_interest)
        high_open_interest_contract = contract
high_open_interest_contract

{   'close_price': '1.8',
    'close_price_date': datetime.date(2026, 2, 6),
    'expiration_date': datetime.date(2026, 2, 10),
    'id': 'c42c03b6-d2f4-49e9-8fc5-0041db160916',
    'name': 'SPY Feb 10 2026 686 Put',
    'open_interest': '11218',
    'open_interest_date': datetime.date(2026, 2, 5),
    'root_symbol': 'SPY',
    'size': '100',
    'status': <AssetStatus.ACTIVE: 'active'>,
    'strike_price': 686.0,
    'style': <ExerciseStyle.AMERICAN: 'american'>,
    'symbol': 'SPY260210P00686000',
    'tradable': True,
    'type': <ContractType.PUT: 'put'>,
    'underlying_asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
    'underlying_symbol': 'SPY'}

In [19]:
# place buy put option order
# - we can place buy put option order same as buy stock/crypto order
req = MarketOrderRequest(
    symbol = high_open_interest_contract.symbol,
    qty = 1,
    side = OrderSide.BUY,
    type = OrderType.MARKET,
    time_in_force = TimeInForce.DAY,
)
res = trade_client.submit_order(req)
res

APIError: {"code":42210000,"message":"options market orders are only allowed during market hours"}

In [ ]:
# get list of orders by specifying option contract symbol
req = GetOrdersRequest(
    status = QueryOrderStatus.ALL,
    symbols = [high_open_interest_contract.symbol],
    limit = 2,
)
orders = trade_client.get_orders(req)
orders

In [ ]:
# below cells should be done after market open otherwise there is no position for the option contract

# get positions filtered by option contract symbol
# if you do this example outside of market hours, you will see empty list
# because we have no position in this option contract
# please wait market open and run this example again
positions = trade_client.get_all_positions()
[pos for pos in positions if pos.symbol == high_open_interest_contract.symbol]

In [ ]:
# get positions by symbol
trade_client.get_open_position(symbol_or_asset_id=high_open_interest_contract.symbol)


In [ ]:
# get positions by contract id
trade_client.get_open_position(symbol_or_asset_id = high_open_interest_contract.id)

In [ ]:
# close the option position
trade_client.close_position(
    symbol_or_asset_id = high_open_interest_contract.symbol,
    close_options = ClosePositionRequest(qty = "1")
)

In [ ]:
# exercise the options position
# - this method does not return anything
trade_client.exercise_options_position(
    symbol_or_contract_id = high_open_interest_contract.symbol
)

# Trade Update (Stream)

With TradingStream client, you can get updates about trades

fyi. you can open this notebook in another window and run below cell to check trade updates.

In [ ]:
# subscribe trade updates
trade_stream_client = TradingStream(api_key, secret_key, paper=paper, url_override = trade_api_wss)

async def trade_updates_handler(data):
    print(data)

trade_stream_client.subscribe_trade_updates(trade_updates_handler)
trade_stream_client.run()

# Market Data (Historical)

In [20]:
# setup option historical data client
option_historical_data_client = OptionHistoricalDataClient(api_key, secret_key, url_override = data_api_url)

In [21]:
# get options historical bars by symbol
req = OptionBarsRequest(
    symbol_or_symbols = high_open_interest_contract.symbol,
    timeframe = TimeFrame(amount = 1, unit = TimeFrameUnit.Hour),   # specify timeframe
    start = now - timedelta(days = 5),                              # specify start datetime, default=the beginning of the current day.
    # end_date=None,                                                # specify end datetime, default=now
    limit = 2,                                                      # specify limit
)
option_historical_data_client.get_option_bars(req).df

open  high   low  close  volume  \
symbol             timestamp                                                    
SPY260210P00686000 2026-02-04 14:00:00+00:00  4.15  4.82  3.39   4.80   278.0   
                   2026-02-04 15:00:00+00:00  5.46  5.65  3.73   4.68   194.0   

                                              trade_count      vwap  
symbol             timestamp                                         
SPY260210P00686000 2026-02-04 14:00:00+00:00         24.0  3.832302  
                   2026-02-04 15:00:00+00:00         47.0  4.772526

In [22]:
# get options historical trades by symbol
req = OptionTradesRequest(
    symbol_or_symbols = high_open_interest_contract.symbol,
    start = now - timedelta(days = 5),                              # specify start datetime, default=the beginning of the current day.
    # end=None,                                                     # specify end datetime, default=now
    limit = 2,                                                      # specify limit
)
option_historical_data_client.get_option_trades(req).df

exchange  price  size  \
symbol             timestamp                                                
SPY260210P00686000 2026-02-04 14:32:02.934585+00:00        Z   4.15   1.0   
                   2026-02-04 14:32:27.210227+00:00        I   4.28  18.0   

                                                    conditions  
symbol             timestamp                                    
SPY260210P00686000 2026-02-04 14:32:02.934585+00:00          f  
                   2026-02-04 14:32:27.210227+00:00          I

In [23]:
# get options exchange codes
option_historical_data_client.get_option_exchange_codes()

{'A': 'AMEX - NYSE American',
 'B': 'BOX - Boston Options Exchange',
 'C': 'CBOE - Cboe Options Exchange',
 'D': 'EMERALD - Miami International Stock Exchange Emerald Options',
 'E': 'EDGX - Cboe EDGX Options Exchange',
 'H': 'GEMX - Nasdaq GEMX',
 'I': 'ISE - Nasdaq International Securities Exchange',
 'J': 'MRX - Nasdaq MRX',
 'M': 'MIAX - Miami International Stock Exchange',
 'N': 'NYSE - NYSE Arca',
 'O': 'OPRA - Options Price Reporting Authority',
 'P': 'PEARL - Miami International Stock Exchange Pearl Options',
 'Q': 'NASD - Nasdaq Options',
 'S': 'SPHR - Miami International Stock Exchange Sapphire Options',
 'T': 'BX - Nasdaq BX Options',
 'U': 'MEMX - Members Options Exchange',
 'W': 'C2 - Cboe C2 Options Exchange',
 'X': 'PHLX - Nasdaq PHLX',
 'Z': 'BATS - Cboe BZX Options Exchange'}

In [24]:
# get option latest quote by symbol
req = OptionLatestQuoteRequest(
    symbol_or_symbols = [high_open_interest_contract.symbol],
)
option_historical_data_client.get_option_latest_quote(req)

{'SPY260210P00686000': {   'ask_exchange': 'U',
     'ask_price': 1.89,
     'ask_size': 31.0,
     'bid_exchange': 'N',
     'bid_price': 1.86,
     'bid_size': 112.0,
     'conditions': 'A',
     'symbol': 'SPY260210P00686000',
     'tape': None,
     'timestamp': datetime.datetime(2026, 2, 6, 20, 59, 59, 820852, tzinfo=TzInfo(UTC))}}

In [25]:
# get option latest trade by symbol
req = OptionLatestTradeRequest(
    symbol_or_symbols = [high_open_interest_contract.symbol],
)
option_historical_data_client.get_option_latest_trade(req)

{'SPY260210P00686000': {   'conditions': 'I',
     'exchange': 'Q',
     'id': None,
     'price': 1.82,
     'size': 111.0,
     'symbol': 'SPY260210P00686000',
     'tape': None,
     'timestamp': datetime.datetime(2026, 2, 6, 21, 14, 55, 995651, tzinfo=TzInfo(UTC))}}

In [26]:
# get option snapshot by symbol
req = OptionSnapshotRequest(
    symbol_or_symbols = [high_open_interest_contract.symbol],
)
option_historical_data_client.get_option_snapshot(req)

{'SPY260210P00686000': {   'greeks': {   'delta': -0.3039,
                   'gamma': 0.0364,
                   'rho': -0.0058,
                   'theta': -1.6588,
                   'vega': 0.1264},
     'implied_volatility': 0.2657,
     'latest_quote': {   'ask_exchange': 'U',
                         'ask_price': 1.89,
                         'ask_size': 31.0,
                         'bid_exchange': 'N',
                         'bid_price': 1.86,
                         'bid_size': 112.0,
                         'conditions': 'A',
                         'symbol': 'SPY260210P00686000',
                         'tape': None,
                         'timestamp': datetime.datetime(2026, 2, 6, 20, 59, 59, 820852, tzinfo=TzInfo(UTC))},
     'latest_trade': {   'conditions': 'I',
                         'exchange': 'Q',
                         'id': None,
                         'price': 1.82,
                         'size': 111.0,
                         'symbol': 'SPY260

In [ ]:
# get option chain by underlying_symbol
req = OptionChainRequest(
    underlying_symbol = high_open_interest_contract.underlying_symbol,
)
option_historical_data_client.get_option_chain(req)

# Market Data (Stream)

In [27]:
option_data_stream_client = OptionDataStream(api_key, secret_key, url_override = option_stream_data_wss)

async def option_data_stream_handler(data):
    print(data)

symbols = [
    high_open_interest_contract.symbol,
]

option_data_stream_client.subscribe_quotes(option_data_stream_handler, *symbols) 
option_data_stream_client.subscribe_trades(option_data_stream_handler, *symbols)

option_data_stream_client.run()

keyboard interrupt, bye


TimeoutError: 